# 形状变换与数组组合

学习目标：按数据含义调整数组形状与轴顺序，选择拼接、堆叠和拆分方法，并核对元素对应关系。

前置知识：数组创建、shape 与轴、基本索引与切片、Python 列表。

运行环境：Python 3.12、NumPy 2.5。

环境准备：[环境配置与运行](README.md)。

工作目录：本 Notebook 所在目录；重启内核后从上到下运行。

示例使用单元内构造的数据，后续单元沿用首次导入的 np。

标明“预期异常”的单元会直接显示原始报错；阅读异常类型与原因后，继续运行下一单元。

## 1 reshape 与数据组织

六个温度读数按“每次两个传感器”的顺序保存时，可以用 reshape() 整理成三行两列。行表示观测，列表示传感器，元素总数不变。

默认的 reshape 按最后一个索引变化最快的顺序读取与排列元素；它不根据业务含义自动识别行列。先确认输入顺序，再给出新形状。

In [1]:
import numpy as np

readings = np.array([20, 21, 22, 23, 24, 25])
table = readings.reshape(3, 2)

print(table)  # 三行依次为 [20 21]、[22 23]、[24 25]。
print(readings.shape, table.shape)  # (6,) (3, 2)。
print(readings.size, table.size)  # 6 6，元素数量相同。
print(table[1, 0])  # 22，第二次观测的第一个传感器。

[[20 21]
 [22 23]
 [24 25]]
(6,) (3, 2)
6 6
22


## 2 维数推断与元素数量

新形状中可以有一个 -1，表示由元素总数和其他轴长推算这一轴的长度。这里 -1 是长度占位值，不是负索引。

六个元素可以整理成两行三列，不能整理成四行两列；无法满足元素数量时触发 ValueError。

In [2]:
values = np.arange(6)

print(values.reshape(2, -1))  # 两行为 [0 1 2]、[3 4 5]，推算列数为 3。
print(values.reshape(-1, 2).shape)  # (3, 2)，推算行数为 3。

[[0 1 2]
 [3 4 5]]
(3, 2)


In [3]:
# 预期 ValueError：6 个元素无法重排为需要 8 个位置的形状 (4, 2)。
values.reshape(4, 2)

ValueError: cannot reshape array of size 6 into shape (4,2)

## 3 轴顺序

### 3.1 二维转置

转置（transpose）交换二维数组的行轴和列轴。transpose() 与二维数组的 T 属性都能完成这一操作：原第 i 行、第 j 列变成结果的第 j 行、第 i 列，i、j 是有效的行、列索引。

如果把同一张两行三列的表变成三行两列，转置与 reshape 会把 20 放到相同位置吗？沿下图分别追踪这个值。

![同一数组转置后第一行是 10、20，按默认顺序 reshape 后第一行是 10、11；两者形状相同。](image/illustration/05-01-transpose-and-reshape.svg)

图中 reshape 使用默认的 C 索引顺序，即先读完一行，再按新形状重新分组。图示只解释元素对应关系，不据此判断是否复制数据。下面的两个输出应分别对应图中的两条路径；不要只检查 shape。

In [4]:
table = np.array([[10, 11, 12], [20, 21, 22]])
transposed = table.T
reshaped = table.reshape(3, 2)

print(transposed)  # 三行为 [10 20]、[11 21]、[12 22]。
print(reshaped)  # 三行为 [10 11]、[12 20]、[21 22]。
print(transposed.shape, reshaped.shape)  # 均为 (3, 2)，但数值位置不同。
print(np.transpose(table)[2, 1], table[1, 2])  # 22 22，对应位置互换。

[[10 20]
 [11 21]
 [12 22]]
[[10 11]
 [12 20]
 [21 22]]
(3, 2) (3, 2)
22 22


### 3.2 多维轴调整

多维数组需要明确新的轴顺序。下面输入形状为 (2, 3, 4)，轴依次表示批次、观测、传感器。

| 函数 | 中文名称／含义 | 本例用途 |
| --- | --- | --- |
| np.transpose | 按给定顺序重排全部轴 | (1, 0, 2) 表示观测、批次、传感器 |
| np.swapaxes | 交换两个轴 | 交换批次与传感器轴 |
| np.moveaxis | 把指定轴移到目标位置 | 把批次轴移到最后，其他轴保持相对顺序 |

不传轴顺序时，transpose 反转全部轴的顺序。一维数组只有一个轴，转置不会把它变成二维列向量。

In [5]:
batches = np.arange(24).reshape(2, 3, 4)
reordered = np.transpose(batches, (1, 0, 2))
swapped = np.swapaxes(batches, 0, 2)
moved = np.moveaxis(batches, 0, -1)

print(reordered.shape)  # (3, 2, 4)：观测、批次、传感器。
print(swapped.shape)  # (4, 3, 2)：传感器、观测、批次。
print(moved.shape)  # (3, 4, 2)：观测、传感器、批次。
print(batches[1, 2, 3], reordered[2, 1, 3], moved[2, 3, 1])  # 都是 23。
print(np.array([10, 20, 30]).T.shape)  # (3,)，转置没有增加轴。

(3, 2, 4)
(4, 3, 2)
(3, 4, 2)
23 23 23
(3,)


## 4 单例轴

### 4.1 增加长度为 1 的轴

单例轴是长度为 1 的轴。np.expand_dims() 在指定位置插入单例轴，与索引中的 np.newaxis 对应。

下面把三项读数分别表示为一行三列和三行一列。新增的是组织数据的轴，元素数不变。

In [6]:
readings = np.array([20, 21, 22])
row = np.expand_dims(readings, axis=0)
column = np.expand_dims(readings, axis=1)

print(row, row.shape)  # [[20 21 22]] (1, 3)。
print(column, column.shape)  # 三行一列，(3, 1)。
print(readings[np.newaxis, :].shape)  # (1, 3)，同样增加首轴。

[[20 21 22]] (1, 3)
[[20]
 [21]
 [22]] (3, 1)
(1, 3)


### 4.2 删除单例轴

np.squeeze() 删除长度为 1 的轴；指定 axis 时只删除该轴，未指定时删除全部单例轴。指定的轴长度不为 1 会触发 ValueError。

当批次轴必须保留时，应明确删除哪个轴，避免把所有单例轴一起删除。

In [7]:
batch = np.array([[[20], [21], [22]]])  # (1, 3, 1)：批次、观测、单个传感器。

print(np.squeeze(batch, axis=2).shape)  # (1, 3)，保留批次轴。
print(np.squeeze(batch).shape)  # (3,)，两个单例轴都被删除。

(1, 3)
(3,)


In [8]:
# 预期 ValueError：axis=1 的长度为 3，squeeze 只能删除长度为 1 的轴。
np.squeeze(batch, axis=1)

ValueError: cannot select an axis to squeeze out which has size not equal to one

## 5 沿已有轴拼接

np.concatenate() 沿已有轴连接数组。除拼接轴外，各轴长度必须一致；拼接不会新增轴。

下面每行是一次观测，每列是一个字段。沿轴 0 拼接增加观测行，沿轴 1 拼接增加字段列。

In [9]:
first = np.array([[1, 2, 3], [4, 5, 6]])
second = np.array([[7, 8, 9], [10, 11, 12]])
more_rows = np.concatenate([first, second], axis=0)
more_columns = np.concatenate([first, second], axis=1)

print(more_rows, more_rows.shape)  # 先 first 后 second 的四行，形状 (4, 3)。
print(more_columns)  # 两行为 [1 2 3 7 8 9]、[4 5 6 10 11 12]。
print(more_columns.shape)  # (2, 6)。

[[ 1  2  3]
 [ 4  5  6]
 [ 7  8  9]
 [10 11 12]] (4, 3)
[[ 1  2  3  7  8  9]
 [ 4  5  6 10 11 12]]
(2, 6)


非拼接轴的长度也需要检查。行数能相加，不代表列数不同的两张表可以直接按行拼接。

In [10]:
first = np.ones((2, 3))
different_columns = np.ones((1, 4))

# 预期 ValueError：沿行拼接时列轴必须一致，这两个数组的列数分别为 3、4。
np.concatenate([first, different_columns], axis=0)

ValueError: all the input array dimensions except for the concatenation axis must match exactly, but along dimension 1, the array at index 0 has size 3 and the array at index 1 has size 4

## 6 沿新轴堆叠

np.stack() 把同形数组沿一个新轴组织起来，输入形状必须完全一致；axis 指定新轴在结果中的位置。两个长度为 3 的数组，既可以表示早晚各一次的三个传感器读数，也可以只接成六个连续读数，选择取决于是否需要保留“早 / 晚”这一层。

![同样的早晚两个一维数组，concatenate 得到形状 6，stack 沿轴 0 和轴 1 分别得到形状 2、3 与 3、2。](image/illustration/05-02-concatenate-and-stack.svg)

蓝色与绿色分别标识早、晚输入。堆叠后的新轴标识输入来自哪一组；拼接沿原轴延长，不会自动保存这个分组轴。

下面先把早晚放在轴 0，再放在轴 1。对照同一个传感器在两个时段的位置，说明两个二维输出各轴的实际含义，再检查拼接的一维结果。

In [11]:
morning = np.array([20, 21, 22])
evening = np.array([23, 24, 25])
by_period = np.stack([morning, evening], axis=0)
by_sensor = np.stack([morning, evening], axis=1)

print(by_period, by_period.shape)  # 两行分别为早晚读数，(2, 3)。
print(by_sensor, by_sensor.shape)  # 三行为 [20 23]、[21 24]、[22 25]，(3, 2)。
print(np.concatenate([morning, evening]).shape)  # (6,)，沿原有一维轴拼接。

[[20 21 22]
 [23 24 25]] (2, 3)
[[20 23]
 [21 24]
 [22 25]] (3, 2)
(6,)


## 7 数组拆分

### 7.1 等分与切分位置

np.split() 返回由子数组组成的列表。第二个参数是整数时，要求指定轴等分成相应份数；传入索引列表时，则在这些位置切开。

下面一维序列有 6 个元素，能等分成 3 份。在位置 2、5 切开时，三份长度分别为 2、3、1。

In [12]:
values = np.arange(6)
equal_parts = np.split(values, 3)
by_position = np.split(values, [2, 5])

print(equal_parts)  # [0 1]、[2 3]、[4 5] 三个数组。
print(by_position)  # [0 1]、[2 3 4]、[5] 三个数组。

[array([0, 1]), array([2, 3]), array([4, 5])]
[array([0, 1]), array([2, 3, 4]), array([5])]


### 7.2 不等分与指定轴

需要尽量均分、但不要求各份长度相同时，可以使用 np.array_split()。不能整除时，靠前的一些子数组会多一个元素。

二维数组也可以按 axis 拆分，轴 0 分行，轴 1 分列。

In [13]:
values = np.arange(7)

# 预期 ValueError：7 个元素不能等分成 3 份。
np.split(values, 3)

ValueError: array split does not result in an equal division

In [14]:
print(np.array_split(values, 3))  # [0 1 2]、[3 4]、[5 6]，长度为 3、2、2。
table = np.arange(8).reshape(2, 4)
left, right = np.split(table, 2, axis=1)
print(left, right)  # 左半为 [[0 1], [4 5]]，右半为 [[2 3], [6 7]]。
print(left.shape, right.shape)  # 都是 (2, 2)。

[array([0, 1, 2]), array([3, 4]), array([5, 6])]
[[0 1]
 [4 5]] [[2 3]
 [6 7]]
(2, 2) (2, 2)


## 8 综合应用：组织观测批次

两批模拟温度各有两次观测，每次三个传感器读数。需要保留批次来源时，用 stack 新增批次轴；需要合成按观测排列的表格时，用 concatenate 延长行轴。

核对两种组织方式中的对应位置，再按批次边界拆回，确认元素含义没有改变。

In [15]:
first = np.array([[20, 21, 22], [23, 24, 25]])
second = np.array([[30, 31, 32], [33, 34, 35]])
batches = np.stack([first, second], axis=0)
observations = np.concatenate([first, second], axis=0)
recovered_first, recovered_second = np.split(observations, 2, axis=0)

print(batches.shape)  # (2, 2, 3)：批次、观测、传感器。
print(observations.shape)  # (4, 3)：按批次先后排列的观测、传感器。
print(batches[1, 0, 2], observations[2, 2])  # 32 32，同一个传感器读数。
print(recovered_first)  # [[20 21 22], [23 24 25]]。
print(recovered_second)  # [[30 31 32], [33 34 35]]。

(2, 2, 3)
(4, 3)
32 32
[[20 21 22]
 [23 24 25]]
[[30 31 32]
 [33 34 35]]


## 9 选学：维数、重复与网格

### 9.1 最低维数

atleast 系列保证输入至少具有指定维数；已有更高维输入不会被降维。它适合接受标量或数组、但输出需要统一最低维数的操作。

| 函数 | 中文名称／含义 |
| --- | --- |
| np.atleast_1d | 至少一维 |
| np.atleast_2d | 至少二维，一维输入变为行数组 |
| np.atleast_3d | 至少三维，一维输入增加首尾单例轴 |

这些函数不推断业务中的行列方向；需要列数组时，仍应明确增加哪个轴。

In [16]:
vector = np.array([10, 20, 30])

print(np.atleast_1d(7).shape)  # (1,)。
print(np.atleast_2d(vector).shape)  # (1, 3)。
print(np.atleast_3d(vector).shape)  # (1, 3, 1)。

(1,)
(1, 3)
(1, 3, 1)


### 9.2 元素重复与整体重复

np.repeat() 重复元素，np.tile() 按给定次数重复整个数组块。两者都增加数据量，应按需要的排列选择。

下面的一维输入适合直接对比：每项连续出现两次，与整段连续出现两次是不同的结果。

In [17]:
values = np.array([1, 2, 3])

print(np.repeat(values, 2))  # [1 1 2 2 3 3]。
print(np.tile(values, 2))  # [1 2 3 1 2 3]。

[1 1 2 2 3 3]
[1 2 3 1 2 3]


### 9.3 边界填充

np.pad() 可以在各轴两端增加元素。下面使用 constant 模式，pad_width 的二元组分别指定一维数组左端与右端的填充数量。

填充改变形状和元素数，不能用来代替保持原数据总量的 reshape。

In [18]:
values = np.array([10, 20, 30])
padded = np.pad(values, (1, 2), mode="constant", constant_values=0)

print(padded)  # [0 10 20 30 0 0]。
print(values.shape, padded.shape)  # (3,) (6,)。

[ 0 10 20 30  0  0]
(3,) (6,)


### 9.4 网格坐标

np.meshgrid() 把各轴的坐标序列组合成网格坐标数组。二维情况下，indexing="ij" 的轴长按输入顺序排列；默认的 indexing="xy" 交换前两个轴对应的长度。

下面 x 有两个位置，y 有三个位置。选择索引约定时，应先决定输出的行列分别代表哪个坐标。

In [19]:
x = np.array([10, 20])
y = np.array([1, 2, 3])
x_ij, y_ij = np.meshgrid(x, y, indexing="ij")
x_xy, y_xy = np.meshgrid(x, y, indexing="xy")

print(x_ij.shape, y_ij.shape)  # 均为 (2, 3)，轴顺序为 x、y。
print(x_xy.shape, y_xy.shape)  # 均为 (3, 2)，行对应 y，列对应 x。
print(x_ij[1, 2], y_ij[1, 2])  # 20 3。
print(x_xy[2, 1], y_xy[2, 1])  # 20 3，同一点使用不同的位置索引。

(2, 3) (2, 3)
(3, 2) (3, 2)
20 3
20 3


## 本章小结

（1）reshape 需要保持元素总数；一个 -1 可推算轴长，转置则重排轴的对应关系。

（2）expand_dims 增加单例轴，squeeze 删除单例轴；先判断哪些轴必须保留。

（3）concatenate 延长已有轴，stack 增加新轴；它们对输入形状的要求不同。

（4）split 按份数等分或按位置拆分，array_split 允许不等长；应能用 shape 和对应位置核对拆分与重组。

## 练习

（1）六个数按每次三个字段排列。整理成二维表格，再转为“字段 × 观测”。打印两次变换后的值与形状，解释 reshape 和转置分别改变了什么。

In [20]:
values = np.array([10, 11, 12, 20, 21, 22])

# 在此整理与转置。
# 检查：先得到 (2, 3)，再得到 (3, 2)；最终第二行为 [11 21]。
# 说明为何直接 reshape(3, 2) 不满足相同的字段对应关系。

（2）两天各记录三个温度。分别满足两个交付要求：一份是保留“天 × 传感器”的二维数组，另一份是按天先后排列的一维序列。选择组合方法并说明理由。

In [21]:
day_one = np.array([18, 19, 20])
day_two = np.array([21, 22, 23])

# 在此写出两种结果与选择理由。
# 检查：第一份为 (2, 3)，第二份为 (6,)，元素顺序与输入一致。
# 再把第一份沿天轴拆成两份，检查各份是否保留天轴。

（3）先预测下面各步的形状，再运行。若批次轴不能删除，说明应如何选择 squeeze 的 axis。

In [22]:
values = np.array([[10, 20, 30]])
expanded = np.expand_dims(values, axis=2)

# 先记录预测，再检查每个单例轴的位置。
print(expanded.shape)
print(np.squeeze(expanded).shape)
print(np.squeeze(expanded, axis=2).shape)
# 在此解释保留批次轴的选择，不只复述输出数字。

(1, 3, 1)
(3,)
(1, 3)


（4）七条观测需要拆给三个批次，各批尽量接近。选择合适的拆分方法，打印每批元素与形状，再按原顺序合并；解释严格等分方法为何不适用。

In [23]:
observations = np.arange(7)

# 在此拆分、查看和重组，并说明选择理由。
# 检查：批次长度为 3、2、2；重组为 [0 1 2 3 4 5 6]，shape 为 (7,)。

### 重点练习提示

对应第（4）题。先独立完成，再按需要查看提示。

（1）检查观测数能否被批次数整除。

（2）用商决定每批的基本长度，用余数决定前面多少批多分一条；最后沿原轴拼接。

### 重点练习参考解析

对应第（4）题。

选择 array_split(observations, 3)，因为 7 不能被 3 整除。7 除以 3 的商为 2、余数为 1，因此第一批为 [0, 1, 2]，后两批为 [3, 4]、[5, 6]，形状分别是 (3,)、(2,)、(2,)。

按返回顺序 concatenate 得到原来的 [0, 1, 2, 3, 4, 5, 6]，形状 (7,)，dtype 不变。split 指定整数 3 时要求等分，会因不可整除而报错；不能通过删去末项来满足本题。

## 参考与引用来源

本章新增示意图由 CMYK Labs 原创，依据下表对应概念与本章教学输入绘制；示意图不作为实际运行截图或数学证明。

| 网站 | 本章参考内容与定位 |
| --- | --- |
| NumPy 官方文档（NumPy 2.5） | [reshape](https://numpy.org/doc/2.5/reference/generated/numpy.reshape.html) 的 shape、order、Notes：元素数量、单个 -1 与默认索引顺序；[transpose](https://numpy.org/doc/2.5/reference/generated/numpy.transpose.html) 的 axes、Notes：轴重排、一维输入与二维转置；[swapaxes](https://numpy.org/doc/2.5/reference/generated/numpy.swapaxes.html)、[moveaxis](https://numpy.org/doc/2.5/reference/generated/numpy.moveaxis.html)：轴参数与相对顺序；[expand_dims](https://numpy.org/doc/2.5/reference/generated/numpy.expand_dims.html)、[squeeze](https://numpy.org/doc/2.5/reference/generated/numpy.squeeze.html)：单例轴、axis、异常与 newaxis 等价示例；[concatenate](https://numpy.org/doc/2.5/reference/generated/numpy.concatenate.html)、[stack](https://numpy.org/doc/2.5/reference/generated/numpy.stack.html)：输入形状和已有轴／新轴；[split](https://numpy.org/doc/2.5/reference/generated/numpy.split.html)、[array_split](https://numpy.org/doc/2.5/reference/generated/numpy.array_split.html)：份数、切分位置、axis 与不等分。选学：[atleast_1d](https://numpy.org/doc/2.5/reference/generated/numpy.atleast_1d.html)、[atleast_2d](https://numpy.org/doc/2.5/reference/generated/numpy.atleast_2d.html)、[atleast_3d](https://numpy.org/doc/2.5/reference/generated/numpy.atleast_3d.html) 的输入维数与输出形状；[repeat](https://numpy.org/doc/2.5/reference/generated/numpy.repeat.html)、[tile](https://numpy.org/doc/2.5/reference/generated/numpy.tile.html)：重复元素与数组块；[pad](https://numpy.org/doc/2.5/reference/generated/numpy.pad.html) 的 pad_width、constant、constant_values；[meshgrid](https://numpy.org/doc/2.5/reference/generated/numpy.meshgrid.html) 的 indexing、Notes：ij 与 xy 的二维形状约定。 |